In [20]:
# Crear y guardar la imagen sintética
from PIL import Image, ImageDraw

# Crear una imagen de 120x120 píxeles con fondo azul oscuro (RGB: 20, 30, 70)
img = Image.new("RGB", (120, 120), color=(20, 30, 70))
draw = ImageDraw.Draw(img)

# Figura 1: Un rectángulo rojo (RGB: 220, 50, 50)
draw.rectangle([15, 15, 55, 55], fill=(220, 50, 50))

# Figura 2: Un círculo verde (RGB: 50, 200, 80)
draw.ellipse([65, 65, 105, 105], fill=(50, 200, 80))

print("Tamaño:", img.size, "| Modo:", img.mode)
img.save("mi_imagen.png")

Tamaño: (120, 120) | Modo: RGB


In [21]:
# Reabrir y verificar píxeles clave
img_reabierta = Image.open("mi_imagen.png")
print("Formato de la imagen:", img_reabierta.format)

# Coordenadas exactas para los píxeles:
px_fondo = img_reabierta.getpixel((5, 5))      # Fondo
px_fig1 = img_reabierta.getpixel((30, 30))     # Rectángulo rojo
px_fig2 = img_reabierta.getpixel((80, 80))     # Círculo verde

print(f"Píxel de fondo en (5, 5): {px_fondo}")
print(f"Píxel de Figura 1 (rojo) en (30, 30): {px_fig1}")
print(f"Píxel de Figura 2 (verde) en (80, 80): {px_fig2}")

Formato de la imagen: PNG
Píxel de fondo en (5, 5): (20, 30, 70)
Píxel de Figura 1 (rojo) en (30, 30): (220, 50, 50)
Píxel de Figura 2 (verde) en (80, 80): (50, 200, 80)


In [22]:
#Transformaciones geométricas y conversión
# 1. Escala de grises con .convert("L")
img_gray = img_reabierta.convert("L")
print("Tamaño tras convert('L'):", img_gray.size)

# 2. Rotación de 90 grados con expand=True
img_rot = img_reabierta.rotate(90, expand=True)
print("Tamaño tras rotate(90, expand=True):", img_rot.size)

# 3. Redimensionar a la mitad (60x60)
img_half = img_reabierta.resize((img_reabierta.width // 2, img_reabierta.height // 2))
print("Tamaño tras resize a la mitad:", img_half.size)

# 4. Volteo vertical (FLIP_TOP_BOTTOM)
img_flip = img_reabierta.transpose(Image.FLIP_TOP_BOTTOM)
print("Tamaño tras flip vertical:", img_flip.size)

Tamaño tras convert('L'): (120, 120)
Tamaño tras rotate(90, expand=True): (120, 120)
Tamaño tras resize a la mitad: (60, 60)
Tamaño tras flip vertical: (120, 120)


In [23]:
#-----------------TAREA 2----------------------

In [24]:
# Realce de brillo con ImageEnhance
from PIL import ImageEnhance, ImageFilter

# Brillo con factor menor a 1.0 (disminuir brillo)
enhancer = ImageEnhance.Brightness(img_reabierta)
img_dark = enhancer.enhance(0.5)

# Brillo con factor mayor a 2.0 (saturación/aumento de brillo)
img_bright = enhancer.enhance(2.5)

# Píxel de la figura 1 (rojo en (30, 30)) ANTES y DESPUÉS
px_antes = img_reabierta.getpixel((30, 30))
px_despues = img_bright.getpixel((30, 30))

print(f"Píxel Figura 1 ANTES del realce (factor 2.5): {px_antes}")
print(f"Píxel Figura 1 DESPUÉS del realce (factor 2.5): {px_despues}")

# Verificación de recorte (clipping en 255)
canal_r_recortado = px_despues[0] == 255
print(f"RECORTE: {canal_r_recortado} ({px_antes[0]} * 2.5 = {px_antes[0]*2.5} -> recortado a {px_despues[0]})")

Píxel Figura 1 ANTES del realce (factor 2.5): (220, 50, 50)
Píxel Figura 1 DESPUÉS del realce (factor 2.5): (255, 125, 125)
RECORTE: True (220 * 2.5 = 550.0 -> recortado a 255)


In [25]:
# Filtros de desenfoque y bordes 
# Aplicar filtro Gaussian Blur
img_blur = img_reabierta.filter(ImageFilter.GaussianBlur(radius=2))
img_blur.save("mi_imagen_blur.png")

# Aplicar filtro Find Edges
img_edges = img_reabierta.filter(ImageFilter.FIND_EDGES)
img_edges.save("mi_imagen_edges.png")

In [26]:
# Conversión a NumPy y comparación de escala de grises
import numpy as np
from skimage import color, filters

# Convertir la imagen PIL a array de NumPy
img_np = np.array(img_reabierta)

# Convertir a escala de grises usando scikit-image (devuelve valores flotantes [0.0, 1.0])
img_sk_gray = color.rgb2gray(img_np)

# Valor del píxel de fondo (5, 5) en scikit-image escalado a [0, 255]
val_sk_fondo = img_sk_gray[5, 5] * 255

# Valor del píxel de fondo (5, 5) en Pillow convert("L")
val_pil_fondo = img_gray.getpixel((5, 5))

print(f"Valor píxel de fondo en Pillow convert('L'): {val_pil_fondo}")
print(f"Valor píxel de fondo en skimage rgb2gray (* 255): {val_sk_fondo:.4f}")

Valor píxel de fondo en Pillow convert('L'): 32
Valor píxel de fondo en skimage rgb2gray (* 255): 30.7590


In [27]:
# Binarización con umbral de Otsu
# Calcular umbral de Otsu con scikit-image
thresh = filters.threshold_otsu(img_sk_gray)
binary_mask = img_sk_gray > thresh

# Proporción de píxeles 'True' (por encima del umbral)
prop_true = np.mean(binary_mask)

print(f"Umbral de Otsu calculado: {thresh:.4f} (equivalente a {thresh*255:.2f} en escala 0-255)")
print(f"Proporción de píxeles True (primer plano): {prop_true:.4f} ({prop_true*100:.2f}%)")

Umbral de Otsu calculado: 0.1216 (equivalente a 31.01 en escala 0-255)
Proporción de píxeles True (primer plano): 0.2079 (20.79%)
